# Lecture 7: Visualizing Data — Line, Scatter, Bar
v.ekc

A table shows you numbers; a chart shows you the *story*. Today we learn the three workhorse charts and, just as important, **which one to use when**. 

**Today's sections:**
1. Census recap
2. Line plots
3. Analysis: males vs. females
4. Scatter plots
5. Bar charts

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Census Recap

Same census table as last time, rebuilt in three tidy steps. (Remember the codes: `SEX` 0/1/2 = total/male/female; `AGE` 999 = all ages.)

In [ ]:
full = Table.read_table('nc-est2019-agesex-res.csv')

In [ ]:
# Keep only the columns we care about
partial = full.select('SEX', 'AGE', 'POPESTIMATE2019')

In [ ]:
# Make things easier to read
us_pop_2019 = partial.relabeled(2, '2019')
us_pop_2019

Three code rows worth knowing, the totals, the babies, the centenarians:

In [ ]:
# totals
us_pop_2019.where('AGE', are.equal_to(999))

In [ ]:
# babies
us_pop_2019.where('AGE', are.equal_to(0))

In [ ]:
# 100+ people
us_pop_2019.where('AGE', are.equal_to(100))

---
## 2. Line Plots

**Line plots** are for numerical values in a natural order, like age, or time. (KL deck, slides 16–19.)

### 📋 Reference Table

| Code | What it does |
|---|---|
| `t.plot('x', 'y')` | line plot of y against x |
| `t.plot('x')` | one line per remaining column |
| `t.scatter('x', 'y')` | scatter plot — two numerical variables |
| `t.barh('category', 'value')` | horizontal bar chart |
| `t.set_format('col', PercentFormatter)` | display a column as percents |
| `t.take(np.arange(a, b))` | rows a through b−1 |

In [ ]:
# Remove the age totals
# AGE 999 is the total count for ages

no_999 = us_pop_2019.where('AGE', are.below(999))

In [ ]:
# SEX 0 is the total counts for MALE and FEMALE
#   and then just have a table of AGE and count of ppl/age

overall = no_999.where('SEX', 0).drop('SEX')
overall

In [ ]:
## Our first line plot! 
overall.plot('AGE', '2019')

**Options for labeling**

In [ ]:
### OPTION 1
# US Population  <--- Just add a comment
overall.plot('AGE', '2019')

In [ ]:
### OPTION 2
overall.plot('AGE', '2019')
print('US Population')  # <--- Print out what it is

In [ ]:
### OPTION 3
overall.plot('AGE', '2019')
plots.title('US Population');    # <--- not needed for Data 111, but a handy skill!

### Try It 1: Peak population 😊

1. Which single age has the **largest** population in `overall`? Answer with code, not squinting.

2. Find that age on the line plot above: does the plot agree?

In [ ]:
# 1. age with the largest population



*2. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Sort descending and read off the top row.*

```python
# 1.
overall.sort('2019', descending=True).column('AGE').item(0)
```

*2. The plot's highest point should sit right at that age, a late-20s/early-30s bump.*

</details>

---
## 3. Analysis: Males vs. Females

One plot is a picture; comparing plots is an **analysis**. Your turn to set it up.

### Try It 2: Build the tables 😊

1. Create a table `males` with columns `AGE` and `2019`, the male population at every age (no totals!).

2. Same for `females`.

In [ ]:
# Fill in the blank: this cell errors until you do! 😅
males = 
males

In [ ]:
# Fill in the blank: this cell errors until you do! 😅
females = 
females

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Filter `no_999` (totals already removed) by the SEX code, then drop the code column.*

```python
# 1.
males = no_999.where('SEX', 1).drop('SEX')

# 2.
females = no_999.where('SEX', 2).drop('SEX')
```

</details>

Run the next cell so the rest of the lecture works:

In [ ]:
# run me: needed below
males = no_999.where('SEX', 1).drop('SEX')
females = no_999.where('SEX', 2).drop('SEX')

In [ ]:
np.arange(0,99)

In [ ]:
males_females_2019 = Table().with_columns(
    'Age', males.column('AGE'),
    'Males', males.column('2019'),
    'Females', females.column('2019')
)
males_females_2019

In [ ]:
# or, create arrays of male and female and construct a table with_columns

# create arrays
males_arr = no_999.where('SEX',1).column('2019')
females_arr = no_999.where('SEX',2).column('2019')

# create Table with_columns from arrays instead of filtered tables!
males_females_2019_2 = Table().with_columns(
    'Age', np.arange(0,101),
    'Males', males_arr,
    'Females', females_arr
)
males_females_2019_2

One table, three columns; `plot('Age')` draws one line per remaining column:

In [ ]:
males_females_2019.plot('Age')

The lines cross; measure the gap as a *proportion* instead:

In [ ]:
# Calculate the percent female for each age
total = males_females_2019.column('Males') + males_females_2019.column('Females')
proportion_female = males_females_2019.column('Females') / total 
proportion_female

In [ ]:
# Add female percent to our table
males_females_2019 = males_females_2019.with_column('Proportion female', proportion_female).set_format('Proportion female',PercentFormatter)
males_females_2019

Now plot the proportion, and then look very carefully at the y-axis:

In [ ]:
males_females_2019.plot('Age', 'Proportion female')

In [ ]:
# ^^ Look at the y-axis! Trend is not as dramatic as you might think

males_females_2019.plot('Age', 'Proportion female')
plots.ylim(0, 1);  # rescale the y axis from 0,1

# actually it still looks dramatic haha! but still check out the YT video by Vox

> **Always check the y-axis!** 😬 The first plot looked dramatic; with the axis starting at 0, the story is much gentler. Same data, honest picture. (KL deck, slides 28–29.)

In [ ]:
males_females_2019.take(np.arange(35,45))

---
## 4. Scatter Plots

When *neither* variable has a natural order, like number of movies vs. total gross, use a **scatter plot**: every dot is one individual.

In [ ]:
# Actors and their highest grossing movies
actors = Table.read_table('actors.csv')
actors

In [ ]:
actors.scatter('Number of Movies', 'Total Gross')

Total gross rewards long careers; average per movie tells a different story:

In [ ]:
actors.scatter('Number of Movies', 'Average per Movie')

In [ ]:
actors.where('Average per Movie', are.above(400))

### Try It 3: Scatter stories 😊

1. Make a scatter plot of `Number of Movies` vs. `Gross` (each actor's **#1 movie**).

2. Which actor has appeared in the **most** movies? Find it with code.

3. Look at your scatter: does being in more movies go with a bigger #1 hit?

In [ ]:
# 1. scatter: Number of Movies vs Gross


In [ ]:
# 2. actor with the most movies


*3. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Any two numerical columns can be scattered; sort + `.item(0)` answers the superlative.*

```python
# 1.
actors.scatter('Number of Movies', 'Gross')

# 2.
actors.sort('Number of Movies', descending=True).column('Actor').item(0)
```

*3. Many right answers, the cloud is fairly flat, so a longer filmography doesn't guarantee a bigger single hit.*

</details>

> **Line vs. scatter:** if the x-values have a natural sequence (age, year), a line plot works. If each row is just an individual, connecting the dots means nothing: scatter it. Guess who that outlier dot is, then check the table above. 🎬

---
## 5. Bar Charts

Categories (studios, titles, flavors) aren't numbers, they get **bar charts**. Watch what happens when we wrongly force a line plot on them:

In [ ]:
# Highest grossing movies as of 2017
top_movies = Table.read_table('top_movies_2017.csv')
top_movies

In [ ]:
# subset the table...
#    use table.take(index/array) to grab the first 10 rows!

top10_adjusted = top_movies.take(np.arange(10))
top10_adjusted

Dollars → millions, for readable labels:

In [ ]:
# Convert to millions of dollars for readability
millions = np.round(top10_adjusted.column('Gross (Adjusted)') / 1000000, 3)
top10_adjusted = top10_adjusted.with_column('Millions', millions)
top10_adjusted

In [ ]:
# A line plot doesn't make sense here: don't do this!
top10_adjusted.plot('Year', 'Millions')

> **Don't do this!** A line plot over `Year` implies a sequence and interpolation that make no sense for 10 individual movies. (KL deck, slides 24–25.) Bar chart to the rescue:

In [ ]:
top10_adjusted.barh('Title', 'Millions')

Bars can show any value column, the movies' ages, for instance:

In [ ]:
# need the top 10 movies
# need the year of the movie as an array
#   2026 year - year of movie --> age of movie as array!
#   add the age of the movie to the table!

top10_adjusted = top10_adjusted.with_column('Age of Movie', 2026 - top10_adjusted.column('Year'))

In [ ]:
top10_adjusted.barh('Title','Age of Movie')

### Try It 4: Unadjusted top ten 😊

1. Build `top10_unadjusted`, the ten movies with the highest **un**adjusted `Gross`.

2. Draw its bar chart of `Gross` by `Title`.

3. Compare with the adjusted top ten above: what kind of movie moves up when you *don't* adjust for inflation?

In [ ]:
# 1. top ten by unadjusted Gross


In [ ]:
# 2. bar chart


*3. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Sort by the other column, take the first ten rows, chart it.*

```python
# 1.
top10_unadjusted = top_movies.sort('Gross', descending=True).take(np.arange(10))

# 2.
top10_unadjusted.barh('Title', 'Gross')
```

*3. Recent blockbusters dominate the unadjusted list: inflation makes modern ticket revenue loooook bigger.*

</details>

---
> **Today's takeaway:** line plots for ordered numbers, scatter plots for pairs of numbers, bar charts for categories, and never trust a plot until you've checked its y-axis. Homework 3 uses all three.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — the three charts
```python
t.plot('x', 'y')        # ordered numbers
t.scatter('x', 'y')     # two numerical variables
t.barh('category', 'value')   # categorical
```